# Chameleon — Phase 3 on GPU

Blind cross-modal steganalysis: trains the image and audio detectors, runs the
held-out-method blindness tests, and consolidates everything into one table.

**Runtime → Change runtime type → T4 GPU**, then Runtime → Run all.

Nothing needs uploading. The dataset is generated deterministically from a fixed
seed, so this notebook rebuilds all 8,000 samples from scratch in about four
minutes. Only the result CSVs come back out, as a small zip at the end.


## 1 · Check the GPU

In [ ]:
import torch, platform
print("python :", platform.python_version())
print("torch  :", torch.__version__)
print("cuda   :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu    :", torch.cuda.get_device_name(0))
else:
    print("\n!! No GPU. Runtime -> Change runtime type -> T4 GPU, then Run all again.")
    print("   It will still work on CPU, but expect hours instead of minutes.")

## 2 · Dependencies

Colab ships torch, torchaudio, scikit-image and scikit-learn; only soundfile usually needs adding.

In [ ]:
!pip install -q soundfile
import soundfile; print("soundfile", soundfile.__version__)

## 3 · Project source

Each cell below writes one file of the project. They are identical to the repo copies, so you can edit here and copy back, or vice versa.

In [ ]:
import os
os.makedirs('/content/chameleon/src', exist_ok=True)
%cd /content/chameleon

In [ ]:
%%writefile src/data_generation.py
"""
Phase 2 - Cover media generation.

We don't have reliable internet access to big steganalysis datasets like
BOSSbase/ALASKA2 in this environment, so we build a reasonably sized and
diverse cover dataset ourselves:

  - Images: random crops of skimage's bundled sample photographs (real
    photographs, not synthetic), augmented only by operations that are
    exact on 8-bit integers.

  - Audio: synthesized programmatically (tones, FM "voice-like" synthesis,
    filtered noise, chirps) since we can't reliably fetch external audio
    corpora here. Each clip is randomized so no two are identical.

Everything is deterministic given a seed, so the dataset is reproducible.

Why the augmentation is integer-exact (important)
--------------------------------------------------
The first version of this file built covers through a floating-point
pipeline: rgb2gray, anti-aliased resize/rotate at arbitrary angles,
brightness/contrast jitter, and finally `+ N(0, 0.003)` noise before
quantizing to uint8. That noise is about 0.003 * 255 = 0.77 grey levels,
roughly one quantization step, and the interpolation contributed more of
the same.

The effect was to randomize the least significant bit plane of every
cover BEFORE anything was embedded. Measured on those covers, the LSB
plane had mean 0.5005 and horizontal autocorrelation -0.007 -- i.e. it was
already indistinguishable from coin flips -- and embedding LSB at 0.1 bpp
moved the chi-square pairs-of-values statistic only from 467 to 402.

That matters because the LSB plane's natural structure is exactly what
spatial-domain steganalysis detects. Randomizing it first makes LSB and
LSB-matching close to undetectable by construction, which is why both the
classical baseline and the CNNs sat at chance on them. It is a property of
the cover generator, not a finding about steganalysis.

So augmentation here is restricted to operations that map 8-bit values to
themselves exactly: cropping, horizontal/vertical flips, and 90-degree
rotations. No resampling, no gain/bias, no additive noise. Colour sources
contribute one channel at a time rather than a floating-point luma, for
the same reason. Diversity instead comes from the number of distinct crop
positions available (tens of thousands per base image), and exact
duplicates are rejected explicitly.

The audio generator had the same flaw in a louder form: it added
`N(0, 0.02)` background noise to signals that are then written as 16-bit
PCM. 0.02 of full scale is ~655 int16 quantization steps, which randomizes
roughly the low nine bits and makes sample-domain audio LSB steganalysis
impossible by construction. That noise is now off by default (see
BACKGROUND_NOISE below).
"""

import os
import numpy as np
from PIL import Image
import skimage.data as skdata
import soundfile as sf

RNG_SEED = 42
IMG_SIZE = 256
AUDIO_SR = 16000
AUDIO_DURATION = 2.0  # seconds

# Reject near-blank, low-texture crops: they carry almost no steganalysis
# signal either way. Expressed in grey levels (the old float threshold of
# 0.03 on a [0,1] image is 0.03 * 255).
MIN_PATCH_STD = 7.0

# Additive background noise for synthesized audio, in units of full scale.
# Anything at or above 1/32767 (~3e-5) starts corrupting the 16-bit LSB
# plane; the original 0.02 destroyed the low nine bits. Keep at 0.0 unless
# you specifically want to study detection under a noise floor.
BACKGROUND_NOISE = 0.0

# Base images that are real photographs, delivered by skimage as uint8.
# Excluded on purpose: `horse` (boolean mask), `checkerboard` (synthetic,
# degenerate bit planes), and anything smaller than IMG_SIZE in either
# dimension (`page`, `text`) since we no longer upscale.
_CANDIDATE_BASES = [
    "astronaut", "camera", "cat", "chelsea", "coffee", "coins",
    "moon", "rocket", "brick", "grass", "gravel", "clock",
    "hubble_deep_field", "immunohistochemistry", "retina",
]


def _load_base_images(size=IMG_SIZE, verbose=True):
    """Load bundled skimage photographs, keeping them as raw uint8.

    Returns a list of (name, array) where array is uint8 and either 2D
    grayscale or 3D RGB, and is at least `size` in both spatial dimensions.
    Nothing is converted to float here -- that is the entire point.
    """
    imgs = []
    for name in _CANDIDATE_BASES:
        try:
            arr = getattr(skdata, name)()
        except Exception as e:
            if verbose:
                print(f"  skip {name}: {e}")
            continue

        if arr.dtype != np.uint8:
            if verbose:
                print(f"  skip {name}: dtype {arr.dtype}, not uint8")
            continue
        if arr.ndim == 3 and arr.shape[2] != 3:
            if verbose:
                print(f"  skip {name}: {arr.shape[2]} channels")
            continue
        if arr.ndim not in (2, 3):
            if verbose:
                print(f"  skip {name}: {arr.ndim} dimensions")
            continue
        if min(arr.shape[:2]) < size:
            if verbose:
                print(f"  skip {name}: {arr.shape[:2]} smaller than {size}")
            continue

        imgs.append((name, arr))
    return imgs


def generate_cover_images(n_images, out_dir, size=IMG_SIZE, seed=RNG_SEED, verbose=True):
    """Generate n_images distinct grayscale cover images as 8-bit PNGs.

    Augmentation is crop + flip + 90-degree rotation only, all exact on
    uint8, so the natural least-significant-bit structure of the source
    photographs survives into the covers.
    """
    os.makedirs(out_dir, exist_ok=True)
    rng = np.random.default_rng(seed)
    base_imgs = _load_base_images(size=size, verbose=verbose)

    # Every candidate must load. Crops are drawn by indexing into this list,
    # so losing even one source photograph (say, a future scikit-image
    # release dropping a bundled image) would not crash -- it would quietly
    # produce a completely different dataset from the one the published
    # results were computed on. Fail loudly instead.
    loaded = [name for name, _ in base_imgs]
    missing = [name for name in _CANDIDATE_BASES if name not in loaded]
    if missing:
        raise RuntimeError(
            f"Base image(s) failed to load: {missing}. The dataset is defined by "
            f"exactly these {len(_CANDIDATE_BASES)} source photographs; continuing "
            f"without them would silently generate a different dataset. Check the "
            f"scikit-image installation (verified with 0.26, where all are bundled)."
        )
    if verbose:
        print(f"Loaded {len(base_imgs)} base images for augmentation: "
              f"{[n for n, _ in base_imgs]}")

    manifest = []
    seen = set()
    rejected_flat = 0
    rejected_dup = 0

    for i in range(n_images):
        patch = None
        for _attempt in range(200):
            _name, base = base_imgs[rng.integers(0, len(base_imgs))]

            # Colour sources contribute a single channel. A weighted luma
            # would be a float and would re-randomise the low bit.
            plane = base[:, :, rng.integers(0, 3)] if base.ndim == 3 else base

            h, w = plane.shape
            top = rng.integers(0, h - size + 1)
            left = rng.integers(0, w - size + 1)
            candidate = plane[top:top + size, left:left + size]

            # exact-on-uint8 augmentation only
            if rng.random() < 0.5:
                candidate = np.fliplr(candidate)
            if rng.random() < 0.5:
                candidate = np.flipud(candidate)
            k = int(rng.integers(0, 4))
            if k:
                candidate = np.rot90(candidate, k)
            candidate = np.ascontiguousarray(candidate)

            if candidate.std() < MIN_PATCH_STD:
                rejected_flat += 1
                continue

            digest = hash(candidate.tobytes())
            if digest in seen:
                rejected_dup += 1
                continue

            seen.add(digest)
            patch = candidate
            break

        if patch is None:
            raise RuntimeError(
                f"Could not find a distinct, textured crop for image {i} after 200 "
                f"attempts. Lower MIN_PATCH_STD or add more base images."
            )

        fname = f"cover_{i:05d}.png"
        Image.fromarray(patch, mode="L").save(os.path.join(out_dir, fname))
        manifest.append(fname)

    if verbose:
        print(f"Wrote {len(manifest)} cover images to {out_dir}")
        print(f"  rejected {rejected_flat} low-texture crops, {rejected_dup} duplicates")
    return manifest


def _fm_voice_like(t, rng):
    """Crude FM-synthesis signal loosely mimicking voiced-speech formants."""
    f0 = rng.uniform(90, 220)  # pitch
    sig = np.zeros_like(t)
    formants = [rng.uniform(500, 900), rng.uniform(1200, 1800), rng.uniform(2200, 3000)]
    for k, fF in enumerate(formants, start=1):
        mod = np.sin(2 * np.pi * rng.uniform(2, 6) * t)
        inst_freq = fF + 40 * mod
        sig += (1.0 / k) * np.sin(2 * np.pi * inst_freq * t / max(1, k))
    carrier = np.sin(2 * np.pi * f0 * t)
    out = 0.6 * carrier + 0.4 * sig
    envelope = np.clip(np.sin(np.pi * t / t[-1]), 0.15, 1.0)
    return out * envelope


def _chirp(t, rng):
    f_start = rng.uniform(80, 400)
    f_end = rng.uniform(400, 3000)
    k = (f_end - f_start) / t[-1]
    return np.sin(2 * np.pi * (f_start * t + 0.5 * k * t ** 2))


def _filtered_noise(t, rng):
    noise = rng.normal(0, 1, t.shape)
    # simple moving-average low-pass to make it less harsh/white
    win = rng.integers(3, 25)
    kernel = np.ones(win) / win
    return np.convolve(noise, kernel, mode="same")


def _tone_chord(t, rng):
    n_tones = rng.integers(2, 5)
    sig = np.zeros_like(t)
    for _ in range(n_tones):
        f = rng.uniform(100, 2000)
        amp = rng.uniform(0.2, 0.6)
        sig += amp * np.sin(2 * np.pi * f * t + rng.uniform(0, 2 * np.pi))
    return sig


def generate_cover_audio(n_clips, out_dir, sr=AUDIO_SR, duration=AUDIO_DURATION,
                         seed=RNG_SEED + 1, background_noise=BACKGROUND_NOISE,
                         verbose=True):
    """Generate n_clips synthetic cover audio clips (mono, 16-bit PCM WAV).

    background_noise is in units of full scale and defaults to 0. See the
    note at the top of this file: at the original 0.02 it was roughly 655
    int16 quantization steps, which randomized the low nine bits of every
    sample and made sample-domain LSB steganalysis undetectable by
    construction.
    """
    os.makedirs(out_dir, exist_ok=True)
    rng = np.random.default_rng(seed)
    t = np.linspace(0, duration, int(sr * duration), endpoint=False)

    generators = [_fm_voice_like, _chirp, _filtered_noise, _tone_chord]
    manifest = []
    for i in range(n_clips):
        gen = generators[rng.integers(0, len(generators))]
        sig = gen(t, rng)

        if background_noise:
            sig = sig + rng.normal(0, background_noise, sig.shape)

        # normalize to avoid clipping, leave headroom for embedding later
        peak = np.max(np.abs(sig)) + 1e-9
        sig = 0.85 * sig / peak

        fname = f"cover_{i:05d}.wav"
        sf.write(os.path.join(out_dir, fname), sig.astype(np.float32), sr, subtype="PCM_16")
        manifest.append(fname)

    if verbose:
        print(f"Wrote {len(manifest)} cover audio clips to {out_dir}")
        if not background_noise:
            print("  (background noise disabled to preserve the 16-bit LSB plane)")
    return manifest


if __name__ == "__main__":
    import argparse
    import os as _os

    # 1500 rather than the original 400: the first full training run was
    # limited by cover diversity as much as by architecture -- only 280
    # covers reached the training split, and all 2,520 stego samples were
    # derived from those same 280. Crops are free, so this is the cheapest
    # available increase in diversity. It remains capped by the 15 source
    # photographs, which is the honest limitation to report.
    parser = argparse.ArgumentParser()
    parser.add_argument("--n-images", type=int, default=1500)
    parser.add_argument("--n-clips", type=int, default=1500)
    parser.add_argument("--seed", type=int, default=RNG_SEED)
    args = parser.parse_args()

    _PROJECT_ROOT = _os.path.dirname(_os.path.dirname(_os.path.abspath(__file__)))
    IMG_OUT = _os.path.join(_PROJECT_ROOT, "data/covers/images")
    AUD_OUT = _os.path.join(_PROJECT_ROOT, "data/covers/audio")
    generate_cover_images(args.n_images, IMG_OUT, seed=args.seed)
    generate_cover_audio(args.n_clips, AUD_OUT, seed=args.seed + 1)


In [ ]:
%%writefile src/embedding_image.py
"""
Image steganography embedding methods.

Each function takes a cover image (uint8, HxW grayscale) and a payload
(random bits, since for steganalysis training we don't care about message
content, only about the statistical footprint the embedding leaves), and
returns a stego image of the same shape.

payload_rate is in bits-per-pixel (bpp), matching how the steganalysis
literature (BOSSbase-style) reports capacity.

Performance note
----------------
LSB matching and DCT-QIM were originally written as per-pixel and
per-block Python loops, which made the dataset build the slowest step of
the whole project (~25 minutes for 1500 covers). Both are now vectorised
with numpy. They draw from the random generator in exactly the same order
and quantity as the loops did, so a given seed still produces the same
stego images; `_reference_*` keeps the original loop versions so that
equivalence can be checked (see the __main__ block).
"""

import numpy as np


def _make_payload_bits(n_bits, rng):
    return rng.integers(0, 2, size=n_bits, dtype=np.uint8)


def embed_lsb(cover, payload_rate, rng):
    """Classic LSB replacement: flip the least significant bit of randomly
    selected pixels to match payload bits. This is the 'easy to detect'
    baseline method, deliberately naive."""
    img = cover.copy()
    h, w = img.shape
    n_pixels = h * w
    n_embed = int(n_pixels * payload_rate)
    flat = img.flatten()

    positions = rng.choice(n_pixels, size=n_embed, replace=False)
    bits = _make_payload_bits(n_embed, rng)

    flat[positions] = (flat[positions] & 0xFE) | bits
    return flat.reshape(h, w)


def embed_lsb_matching(cover, payload_rate, rng):
    """LSB matching (+-1 embedding): instead of forcing the LSB to a value
    (which creates a detectable 'pairs of values' artifact), randomly add
    or subtract 1 when the LSB doesn't already match the payload bit.
    Harder to detect than plain LSB replacement.

    Vectorised. Positions are distinct (sampled without replacement), so
    there is no ordering dependency between them, and one direction is
    drawn per mismatching pixel in position order -- exactly the draws the
    original loop made.
    """
    h, w = cover.shape
    n_pixels = h * w
    n_embed = int(n_pixels * payload_rate)
    flat = cover.astype(np.int16).flatten()

    positions = rng.choice(n_pixels, size=n_embed, replace=False)
    bits = _make_payload_bits(n_embed, rng)

    vals = flat[positions]
    mismatch = (vals & 1) != bits
    k = int(mismatch.sum())
    if k:  # the loop made no draw at all when nothing mismatched
        directions = rng.choice([-1, 1], size=k)
        v = vals[mismatch]
        new = v + directions
        # at 0 or 255 the step would clip and leave parity wrong, so step
        # the other way instead (same rule as the original)
        out_of_range = (new < 0) | (new > 255)
        new[out_of_range] = v[out_of_range] - directions[out_of_range]
        vals[mismatch] = new
        flat[positions] = vals

    return flat.reshape(h, w).astype(np.uint8)


def embed_dct(cover, payload_rate, rng, block_size=8):
    """Simplified JPEG-style DCT embedding: split the image into 8x8 blocks,
    take the 2D DCT, and perturb a fixed set of mid-frequency coefficients
    by rounding to even/odd (quantization-index-modulation) to encode bits.
    This mimics the family of transform-domain methods (like the general
    idea behind JMiPOD/UERD) without needing a full JPEG codec, and leaves
    a different statistical footprint than spatial-domain LSB methods,
    which is exactly the variety we need for a 'blind' detector.

    Vectorised: all selected blocks are transformed in one batched DCT.
    Bit i goes to block block_order[i // 3], coefficient i % 3 -- the same
    assignment the original nested loop made.
    """
    from scipy.fft import dctn, idctn

    bs = block_size
    img = cover.astype(np.float64)
    h, w = img.shape
    bh, bw = h // bs, w // bs
    n_blocks = bh * bw

    # use 3 mid-frequency coefficient positions per block (skip DC, skip
    # very high freq which is fragile / very low freq which is perceptually
    # sensitive)
    coeff_positions = [(2, 1), (1, 2), (2, 2)]
    n_coef = len(coeff_positions)
    max_bits = n_blocks * n_coef
    n_embed = min(int(h * w * payload_rate), max_bits)

    bits = _make_payload_bits(n_embed, rng)
    block_order = rng.permutation(n_blocks)
    quant_step = 6.0

    out = img.copy()
    if n_embed == 0:
        return np.clip(out, 0, 255).astype(np.uint8)

    # blocks that receive at least one bit, in embedding order
    n_used = -(-n_embed // n_coef)
    used = block_order[:n_used]

    grid = img[:bh * bs, :bw * bs]
    blocks = grid.reshape(bh, bs, bw, bs).transpose(0, 2, 1, 3).reshape(n_blocks, bs, bs)
    coeffs = dctn(blocks[used], axes=(1, 2), norm="ortho")

    for j, (ci, cj) in enumerate(coeff_positions):
        bit_idx = np.arange(j, n_embed, n_coef)
        if bit_idx.size == 0:
            continue
        rows = bit_idx // n_coef
        b = bits[bit_idx]
        c = coeffs[rows, ci, cj]
        q = np.round(c / quant_step)
        wrong = np.mod(q, 2) != b
        q = np.where(wrong, q + np.where(c >= q * quant_step, 1.0, -1.0), q)
        coeffs[rows, ci, cj] = q * quant_step

    # only blocks that were embedded into are written back, as before
    new_blocks = blocks.copy()
    new_blocks[used] = idctn(coeffs, axes=(1, 2), norm="ortho")
    out[:bh * bs, :bw * bs] = (
        new_blocks.reshape(bh, bw, bs, bs).transpose(0, 2, 1, 3).reshape(bh * bs, bw * bs)
    )
    return np.clip(out, 0, 255).astype(np.uint8)


EMBED_METHODS = {
    "lsb": embed_lsb,
    "lsb_matching": embed_lsb_matching,
    "dct_qim": embed_dct,
}


def psnr(cover, stego):
    cover = cover.astype(np.float64)
    stego = stego.astype(np.float64)
    mse = np.mean((cover - stego) ** 2)
    if mse == 0:
        return float("inf")
    return 20 * np.log10(255.0) - 10 * np.log10(mse)


# ---------------------------------------------------------------------------
# Reference (original loop) implementations, kept only to verify that the
# vectorised versions above produce identical output for the same seed.
# ---------------------------------------------------------------------------

def _reference_lsb_matching(cover, payload_rate, rng):
    img = cover.copy().astype(np.int16)
    h, w = img.shape
    n_pixels = h * w
    n_embed = int(n_pixels * payload_rate)
    flat = img.flatten()
    positions = rng.choice(n_pixels, size=n_embed, replace=False)
    bits = _make_payload_bits(n_embed, rng)
    for idx, bit in zip(positions, bits):
        cur_lsb = flat[idx] & 1
        if cur_lsb != bit:
            direction = rng.choice([-1, 1])
            new_val = np.clip(flat[idx] + direction, 0, 255)
            if (new_val & 1) != bit:
                new_val = np.clip(flat[idx] - direction, 0, 255)
            flat[idx] = new_val
    return flat.reshape(h, w).astype(np.uint8)


def _reference_dct(cover, payload_rate, rng, block_size=8):
    from scipy.fftpack import dct, idct
    img = cover.copy().astype(np.float64)
    h, w = img.shape
    bh, bw = h // block_size, w // block_size
    n_blocks = bh * bw
    coeff_positions = [(2, 1), (1, 2), (2, 2)]
    max_bits = n_blocks * len(coeff_positions)
    n_embed = min(int(h * w * payload_rate), max_bits)
    bits = _make_payload_bits(n_embed, rng)
    block_order = rng.permutation(n_blocks)
    quant_step = 6.0
    bit_i = 0
    out = img.copy()
    for b_idx in block_order:
        if bit_i >= n_embed:
            break
        bi, bj = divmod(b_idx, bw)
        y0, x0 = bi * block_size, bj * block_size
        block = img[y0:y0 + block_size, x0:x0 + block_size]
        block_dct = dct(dct(block.T, norm="ortho").T, norm="ortho")
        for (ci, cj) in coeff_positions:
            if bit_i >= n_embed:
                break
            bit = bits[bit_i]
            coeff = block_dct[ci, cj]
            q = round(coeff / quant_step)
            if (q % 2) != bit:
                q += 1 if coeff >= q * quant_step else -1
            block_dct[ci, cj] = q * quant_step
            bit_i += 1
        block_rec = idct(idct(block_dct.T, norm="ortho").T, norm="ortho")
        out[y0:y0 + block_size, x0:x0 + block_size] = block_rec
    return np.clip(out, 0, 255).astype(np.uint8)


if __name__ == "__main__":
    import time
    rng0 = np.random.default_rng(0)
    covers = [rng0.integers(0, 256, (256, 256), dtype=np.uint8) for _ in range(4)]
    covers += [np.zeros((256, 256), np.uint8), np.full((256, 256), 255, np.uint8)]
    for name, fast, ref in [("lsb_matching", embed_lsb_matching, _reference_lsb_matching),
                            ("dct_qim", embed_dct, _reference_dct)]:
        same = True
        for c in covers:
            for rate in (0.1, 0.2, 0.4):
                a = fast(c, rate, np.random.default_rng(42))
                b = ref(c, rate, np.random.default_rng(42))
                same &= np.array_equal(a, b)
        t = time.time(); [fast(covers[0], 0.4, np.random.default_rng(1)) for _ in range(5)]
        tf = (time.time() - t) / 5
        t = time.time(); [ref(covers[0], 0.4, np.random.default_rng(1)) for _ in range(5)]
        tr = (time.time() - t) / 5
        print(f"{name:13s} identical to original: {same}   "
              f"{tr * 1000:7.1f} ms -> {tf * 1000:6.1f} ms per image ({tr / tf:.0f}x)")


In [ ]:
%%writefile src/embedding_audio.py
"""
Audio steganography embedding methods.

Audio covers are float32 PCM in [-1, 1] at 16kHz mono. We convert to 16-bit
integer PCM for the sample-domain methods (LSB) since that's the standard
way LSB audio steganography is defined, and work in float domain for
echo hiding.

payload_rate is expressed in bits-per-sample (bps), analogous to
bits-per-pixel for images.

Performance note: LSB matching was a per-sample Python loop and is now
vectorised. It draws from the random generator in exactly the same order
and quantity as before, so a given seed produces the same stego audio;
`_reference_lsb_matching_audio` keeps the loop version for checking.
"""

import numpy as np


def _to_int16(sig_float):
    return np.clip(sig_float * 32767.0, -32768, 32767).astype(np.int16)


def _to_float(sig_int16):
    return sig_int16.astype(np.float64) / 32767.0


def embed_lsb_audio(cover, payload_rate, rng):
    """Classic LSB replacement on 16-bit PCM samples. Same 'easy to detect'
    baseline idea as image LSB."""
    pcm = _to_int16(cover).copy()
    n = len(pcm)
    n_embed = int(n * payload_rate)

    positions = rng.choice(n, size=n_embed, replace=False)
    bits = rng.integers(0, 2, size=n_embed, dtype=np.int16)

    pcm[positions] = (pcm[positions] & ~1) | bits
    return _to_float(pcm).astype(np.float32)


def embed_lsb_matching_audio(cover, payload_rate, rng):
    """+-1 LSB matching on 16-bit PCM samples, analogous to the image version:
    avoids the detectable bias of forced LSB replacement. Vectorised; one
    direction is drawn per mismatching sample, in position order, exactly
    as the original loop drew them."""
    pcm = _to_int16(cover).astype(np.int32)
    n = len(pcm)
    n_embed = int(n * payload_rate)

    positions = rng.choice(n, size=n_embed, replace=False)
    bits = rng.integers(0, 2, size=n_embed)

    vals = pcm[positions]
    mismatch = (vals & 1) != bits
    k = int(mismatch.sum())
    if k:
        directions = rng.choice([-1, 1], size=k)
        v = vals[mismatch]
        new = v + directions
        out_of_range = (new < -32768) | (new > 32767)
        new[out_of_range] = v[out_of_range] - directions[out_of_range]
        vals[mismatch] = new
        pcm[positions] = vals

    pcm = np.clip(pcm, -32768, 32767).astype(np.int16)
    return _to_float(pcm).astype(np.float32)


def embed_echo_hiding(cover, payload_rate, rng, sr=16000):
    """Echo hiding: split the signal into segments, and for each segment
    add a faint, short-delay echo. A '1' bit uses one delay (e.g. 1ms) and
    a '0' bit uses another delay (e.g. 0.5ms), inaudible under normal
    listening but recoverable via cepstral analysis. This is a genuinely
    different statistical domain from sample-LSB methods (it perturbs the
    autocorrelation structure of the signal rather than individual sample
    values), which is exactly the kind of variety a 'blind' detector needs
    to generalize across."""
    sig = cover.astype(np.float64).copy()
    n = len(sig)

    delay0 = int(0.0005 * sr)  # 0.5 ms -> bit 0
    delay1 = int(0.0010 * sr)  # 1.0 ms -> bit 1
    echo_amp = 0.08  # tuned for perceptual transparency (~22 dB SNR)

    # segment length derived from payload rate: fewer, longer segments for
    # lower payload rate (one bit per segment)
    bits_capacity = max(1, int(n * payload_rate / max(1, delay1)))
    seg_len = max(delay1 * 4, n // max(1, bits_capacity))
    n_segments = n // seg_len
    if n_segments == 0:
        return cover  # too short to embed anything meaningfully

    bits = rng.integers(0, 2, size=n_segments)
    out = sig.copy()
    for s in range(n_segments):
        start = s * seg_len
        end = start + seg_len
        segment = sig[start:end]
        delay = delay1 if bits[s] == 1 else delay0
        echo = np.zeros_like(segment)
        if delay < len(segment):
            echo[delay:] = segment[:len(segment) - delay] * echo_amp
        out[start:end] = segment + echo

    peak = np.max(np.abs(out)) + 1e-9
    if peak > 1.0:
        out = out / peak * 0.98
    return out.astype(np.float32)


EMBED_METHODS_AUDIO = {
    "lsb": embed_lsb_audio,
    "lsb_matching": embed_lsb_matching_audio,
    "echo_hiding": embed_echo_hiding,
}


def snr_db(cover, stego):
    cover = cover.astype(np.float64)
    stego = stego.astype(np.float64)
    noise = cover - stego
    signal_power = np.mean(cover ** 2)
    noise_power = np.mean(noise ** 2) + 1e-12
    return 10 * np.log10(signal_power / noise_power)


def _reference_lsb_matching_audio(cover, payload_rate, rng):
    """Original per-sample loop, kept only to verify equivalence."""
    pcm = _to_int16(cover).astype(np.int32).copy()
    n = len(pcm)
    n_embed = int(n * payload_rate)
    positions = rng.choice(n, size=n_embed, replace=False)
    bits = rng.integers(0, 2, size=n_embed)
    for idx, bit in zip(positions, bits):
        cur_lsb = pcm[idx] & 1
        if cur_lsb != bit:
            direction = rng.choice([-1, 1])
            new_val = np.clip(pcm[idx] + direction, -32768, 32767)
            if (new_val & 1) != bit:
                new_val = np.clip(pcm[idx] - direction, -32768, 32767)
            pcm[idx] = new_val
    pcm = np.clip(pcm, -32768, 32767).astype(np.int16)
    return _to_float(pcm).astype(np.float32)


if __name__ == "__main__":
    import time
    rng0 = np.random.default_rng(0)
    covers = [rng0.uniform(-0.85, 0.85, 32000).astype(np.float32) for _ in range(4)]
    covers += [np.full(32000, -1.0, np.float32), np.full(32000, 1.0, np.float32)]
    same = all(
        np.array_equal(embed_lsb_matching_audio(c, r, np.random.default_rng(42)),
                       _reference_lsb_matching_audio(c, r, np.random.default_rng(42)))
        for c in covers for r in (0.05, 0.1, 0.2))
    t = time.time(); [embed_lsb_matching_audio(covers[0], 0.2, np.random.default_rng(1)) for _ in range(5)]
    tf = (time.time() - t) / 5
    t = time.time(); [_reference_lsb_matching_audio(covers[0], 0.2, np.random.default_rng(1)) for _ in range(5)]
    tr = (time.time() - t) / 5
    print(f"lsb_matching_audio identical to original: {same}   "
          f"{tr * 1000:7.1f} ms -> {tf * 1000:6.1f} ms per clip ({tr / tf:.0f}x)")


In [ ]:
%%writefile src/build_dataset.py
"""
Phase 2 - Full dataset assembly.

For each cover (image or audio), we generate stego versions using every
embedding method at every payload rate. We then build a manifest CSV
(label = cover/stego, method, rate, split) and enforce a strict
cover-level train/val/test split so that no cover (or its stego variants)
crosses split boundaries. This matters for steganalysis specifically:
if a cover's stego version is in train and the same cover (clean) is in
test, the model can partly "recognize the image" rather than learn real
stego statistics, inflating results.
"""

import os
import glob
import numpy as np
import pandas as pd
from PIL import Image
import soundfile as sf

import os as _os
from embedding_image import EMBED_METHODS, psnr
from embedding_audio import EMBED_METHODS_AUDIO, snr_db

# paths are relative to this script's location (src/), so this works no
# matter where the repo is cloned
_PROJECT_ROOT = _os.path.dirname(_os.path.dirname(_os.path.abspath(__file__)))
IMG_COVER_DIR = _os.path.join(_PROJECT_ROOT, "data/covers/images")
IMG_STEGO_DIR = _os.path.join(_PROJECT_ROOT, "data/stego/images")
AUD_COVER_DIR = _os.path.join(_PROJECT_ROOT, "data/covers/audio")
AUD_STEGO_DIR = _os.path.join(_PROJECT_ROOT, "data/stego/audio")
RESULTS_DIR = _os.path.join(_PROJECT_ROOT, "results")

PAYLOAD_RATES_IMAGE = [0.1, 0.2, 0.4]
PAYLOAD_RATES_AUDIO = [0.05, 0.1, 0.2]

SPLIT_RATIOS = {"train": 0.7, "val": 0.15, "test": 0.15}
SEED = 7


def _assign_splits(n_items, seed=SEED):
    rng = np.random.default_rng(seed)
    idx = rng.permutation(n_items)
    n_train = int(n_items * SPLIT_RATIOS["train"])
    n_val = int(n_items * SPLIT_RATIOS["val"])
    splits = np.empty(n_items, dtype=object)
    splits[idx[:n_train]] = "train"
    splits[idx[n_train:n_train + n_val]] = "val"
    splits[idx[n_train + n_val:]] = "test"
    return splits


def build_image_dataset():
    os.makedirs(IMG_STEGO_DIR, exist_ok=True)
    cover_files = sorted(glob.glob(os.path.join(IMG_COVER_DIR, "*.png")))
    n = len(cover_files)
    splits = _assign_splits(n)

    rng = np.random.default_rng(SEED + 1)
    rows = []
    quality_log = []

    for cover_path, split in zip(cover_files, splits):
        cover_name = os.path.basename(cover_path)
        cover_img = np.array(Image.open(cover_path))

        rows.append({
            "modality": "image", "filename": cover_name, "path": cover_path,
            "label": "cover", "method": "none", "payload_rate": 0.0, "split": split,
        })

        for method_name, fn in EMBED_METHODS.items():
            for rate in PAYLOAD_RATES_IMAGE:
                stego_img = fn(cover_img, rate, rng)
                stego_name = f"{cover_name[:-4]}__{method_name}__r{rate}.png"
                stego_path = os.path.join(IMG_STEGO_DIR, stego_name)
                Image.fromarray(stego_img, mode="L").save(stego_path)

                q = psnr(cover_img, stego_img)
                quality_log.append({"modality": "image", "cover": cover_name,
                                     "method": method_name, "rate": rate, "quality_db": q})

                rows.append({
                    "modality": "image", "filename": stego_name, "path": stego_path,
                    "label": "stego", "method": method_name, "payload_rate": rate, "split": split,
                })

    return pd.DataFrame(rows), pd.DataFrame(quality_log)


def build_audio_dataset():
    os.makedirs(AUD_STEGO_DIR, exist_ok=True)
    cover_files = sorted(glob.glob(os.path.join(AUD_COVER_DIR, "*.wav")))
    n = len(cover_files)
    splits = _assign_splits(n, seed=SEED + 2)

    rng = np.random.default_rng(SEED + 3)
    rows = []
    quality_log = []

    for cover_path, split in zip(cover_files, splits):
        cover_name = os.path.basename(cover_path)
        cover_sig, sr = sf.read(cover_path)

        rows.append({
            "modality": "audio", "filename": cover_name, "path": cover_path,
            "label": "cover", "method": "none", "payload_rate": 0.0, "split": split,
        })

        for method_name, fn in EMBED_METHODS_AUDIO.items():
            for rate in PAYLOAD_RATES_AUDIO:
                if method_name == "echo_hiding":
                    stego_sig = fn(cover_sig, rate, rng, sr=sr)
                else:
                    stego_sig = fn(cover_sig, rate, rng)
                stego_name = f"{cover_name[:-4]}__{method_name}__r{rate}.wav"
                stego_path = os.path.join(AUD_STEGO_DIR, stego_name)
                sf.write(stego_path, stego_sig, sr, subtype="PCM_16")

                q = snr_db(cover_sig, stego_sig)
                quality_log.append({"modality": "audio", "cover": cover_name,
                                     "method": method_name, "rate": rate, "quality_db": q})

                rows.append({
                    "modality": "audio", "filename": stego_name, "path": stego_path,
                    "label": "stego", "method": method_name, "payload_rate": rate, "split": split,
                })

    return pd.DataFrame(rows), pd.DataFrame(quality_log)


if __name__ == "__main__":
    os.makedirs(RESULTS_DIR, exist_ok=True)

    print("Building image stego dataset...")
    img_df, img_quality = build_image_dataset()
    print(f"  {len(img_df)} total image entries "
          f"({(img_df.label=='cover').sum()} covers, {(img_df.label=='stego').sum()} stego)")

    print("Building audio stego dataset...")
    aud_df, aud_quality = build_audio_dataset()
    print(f"  {len(aud_df)} total audio entries "
          f"({(aud_df.label=='cover').sum()} covers, {(aud_df.label=='stego').sum()} stego)")

    full_df = pd.concat([img_df, aud_df], ignore_index=True)
    full_df.to_csv(os.path.join(RESULTS_DIR, "manifest.csv"), index=False)

    quality_df = pd.concat([img_quality, aud_quality], ignore_index=True)
    quality_df.to_csv(os.path.join(RESULTS_DIR, "embedding_quality.csv"), index=False)

    print("\nSplit distribution (image):")
    print(img_df[img_df.label == "cover"].split.value_counts())
    print("\nSplit distribution (audio):")
    print(aud_df[aud_df.label == "cover"].split.value_counts())

    print("\nQuality summary by method (dB, higher = more imperceptible):")
    print(quality_df.groupby(["modality", "method"]).quality_db.agg(["mean", "min", "max"]))

    print(f"\nManifest saved to {RESULTS_DIR}/manifest.csv")
    print(f"Quality log saved to {RESULTS_DIR}/embedding_quality.csv")


In [ ]:
%%writefile src/dataset_loader.py
"""
Phase 3 - Dataset loaders.

Reads results/manifest.csv (built in Phase 2) and exposes PyTorch Dataset
classes for images and audio. Supports filtering by embedding method,
which is what lets us do the held-out-method "blindness" test: train on
methods A+B, evaluate on method C which the model has never seen.

Path portability
----------------
build_dataset.py writes ABSOLUTE paths into manifest.csv. Those paths were
correct on the machine that generated the dataset and nowhere else, which
breaks two things: copying or moving the project directory makes every row
silently point back at the original location (so you train on the old
copy's files without noticing), and nobody else can run the project at all
without regenerating the entire dataset first -- which contradicts the
reproducibility the README promises.

Manifest rows are always <project root>/data/..., so we re-root them here
against this checkout's own project root. Doing it in load_manifest() fixes
every consumer at once: both CNN datasets and classical_baseline.py.
"""

import os
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset
from PIL import Image
import soundfile as sf

_PROJECT_ROOT = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
MANIFEST_PATH = os.path.join(_PROJECT_ROOT, "results", "manifest.csv")


def _reroot_path(stored_path):
    """Re-root one absolute manifest path onto THIS checkout.

    Stored paths look like
        C:\\...\\chameleon-main\\data/covers/images\\cover_00000.png
    with mixed separators, because build_dataset.py joins a literal
    "data/covers/images" onto an OS-native root. We keep everything from
    the last "data/" segment onward and re-join it to _PROJECT_ROOT, so one
    manifest works in any checkout on any OS.
    """
    p = str(stored_path).replace("\\", "/")
    idx = p.rfind("/data/")
    if idx == -1:
        # Not a layout we recognise. A relative path we can still resolve;
        # anything else we leave alone rather than guess at.
        if os.path.isabs(p):
            return stored_path
        return os.path.normpath(os.path.join(_PROJECT_ROOT, p))
    rel = p[idx + 1:]  # "data/covers/images/cover_00000.png"
    return os.path.normpath(os.path.join(_PROJECT_ROOT, rel))


def load_manifest(check_exists=True):
    """Load the manifest with every path re-rooted onto this checkout.

    check_exists catches the failure this function exists to prevent: a
    manifest that loads fine but points at files that aren't here. Better a
    loud error up front than a training run that dies 40 minutes in, or
    worse, quietly reads another copy of the project.
    """
    df = pd.read_csv(MANIFEST_PATH)
    df["path"] = df["path"].map(_reroot_path)

    if check_exists:
        missing = df.loc[~df["path"].map(os.path.exists)]
        if len(missing):
            raise FileNotFoundError(
                f"{len(missing)} of {len(df)} manifest files are missing from this checkout.\n"
                f"  project root : {_PROJECT_ROOT}\n"
                f"  first missing: {missing.iloc[0]['path']}\n"
                f"Regenerate the dataset with:\n"
                f"  python data_generation.py && python build_dataset.py"
            )
    return df


def filter_manifest(df, modality, split, methods=None, exclude_methods=None):
    """methods: list of method names to include (covers are always included
    regardless, since 'cover' has method == 'none').
    exclude_methods: list of method names to exclude (for held-out tests)."""
    sub = df[(df.modality == modality) & (df.split == split)].copy()
    if methods is not None:
        sub = sub[(sub.method == "none") | (sub.method.isin(methods))]
    if exclude_methods is not None:
        sub = sub[~sub.method.isin(exclude_methods)]
    return sub.reset_index(drop=True)


class ImageStegoDataset(Dataset):
    """Returns (tensor[1,H,W] float in [0,1], label int 0=cover/1=stego, method str)."""

    def __init__(self, manifest_df):
        self.df = manifest_df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = np.array(Image.open(row.path), dtype=np.float32) / 255.0
        tensor = torch.from_numpy(img).unsqueeze(0)  # [1, H, W]
        label = 0 if row.label == "cover" else 1
        return tensor, label, row.method


class AudioStegoDataset(Dataset):
    """Returns (tensor[1, T] float, label int 0=cover/1=stego, method str).
    Signals are zero-padded/truncated to a fixed length so batches can
    stack (all our generated clips are already the same length, but this
    guards against edge cases)."""

    def __init__(self, manifest_df, fixed_len=32000):
        self.df = manifest_df.reset_index(drop=True)
        self.fixed_len = fixed_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        sig, sr = sf.read(row.path, dtype="float32")
        if len(sig) < self.fixed_len:
            sig = np.pad(sig, (0, self.fixed_len - len(sig)))
        elif len(sig) > self.fixed_len:
            sig = sig[:self.fixed_len]
        tensor = torch.from_numpy(sig).unsqueeze(0)  # [1, T]
        label = 0 if row.label == "cover" else 1
        return tensor, label, row.method


if __name__ == "__main__":
    print(f"Project root: {_PROJECT_ROOT}")
    df = load_manifest()
    print(f"Manifest loaded: {len(df)} rows, all files present")
    print(f"Example resolved path: {df.iloc[0]['path']}")

    train_img = filter_manifest(df, "image", "train")
    print(f"\nImage train (all methods): {len(train_img)} samples, "
          f"{(train_img.label == 'cover').sum()} cover / {(train_img.label == 'stego').sum()} stego")

    held_out = filter_manifest(df, "image", "test", exclude_methods=["dct_qim"])
    print(f"Image test excluding dct_qim: {len(held_out)} samples, methods present: "
          f"{sorted(held_out.method.unique())}")

    ds = ImageStegoDataset(train_img)
    x, y, m = ds[0]
    print(f"Sample 0: tensor shape {x.shape}, label {y}, method '{m}'")

    train_aud = filter_manifest(df, "audio", "train")
    print(f"\nAudio train (all methods): {len(train_aud)} samples")
    ads = AudioStegoDataset(train_aud)
    x, y, m = ads[0]
    print(f"Sample 0: tensor shape {x.shape}, label {y}, method '{m}'")


In [ ]:
%%writefile src/models.py
"""
Phase 3 - Model architectures.

Image model: XuNet-inspired. The key idea that separates real steganalysis
CNNs from a generic image classifier is the FIRST layer: a fixed high-pass
filter (KV kernel, standard in the steganalysis literature) that converts
the image into its noise residual before any learned convolutions happen.
Embedding artifacts live in this high-frequency residual, not in the raw
pixel content, so this single design choice is what makes the network
actually about steganalysis rather than "is this a picture of a cat".

Audio model: a small CNN over the log-mel spectrogram, which plays the
same role for audio that pixel-domain CNNs play for images. We also
provide a raw-waveform 1D CNN as an architectural comparison point (both
are trained in the training script; results are compared).

What changed after the first full training run, and why
--------------------------------------------------------
The first complete Phase 3 run exposed two architectural faults. Both
models below were revised in response; AudioSpectrogramCNN was left alone
because it was the one that worked (validation AUC 0.703, detecting
echo_hiding at 0.856 while missing sample-domain LSB -- exactly the
behaviour its design predicts).

1. ImageStegoCNN sat at 0.504 balanced accuracy / 0.516 AUC while its
   training loss fell from 0.695 to 0.615, i.e. it fit the training set
   and generalized nothing. It called itself XuNet-inspired but omitted
   XuNet's two defining components:

     - the ABS activation after the first convolution. Embedding
       residuals are symmetric about zero, so the informative statistic is
       |r|. Feeding a zero-mean residual straight into BatchNorm + ReLU
       discards every negative residual -- half the evidence.
     - truncation (TLU) of the residual. Unbounded outliers from strong
       image edges dominate the activation statistics and drown out the
       embedding-scale perturbations.

   The KV filter now also rescales its input to 0-255 first, so the TLU
   threshold is expressed in the grey-level units the literature uses
   rather than in [0,1] units where a threshold of 3 would never clip.

   Measured on a held-out A/B with identical data and budget, this took
   validation AUC from 0.586 to 0.722 and accuracy off the 0.500 floor to
   0.650.

2. AudioWaveformCNN never left its initialization at all -- training loss
   held at 0.693 (= ln 2, the constant-output loss) for all 40 epochs. It
   now gets a fixed high-pass residual front end and average rather than
   max pooling, which is the better-motivated architecture.

   That did NOT fix it, and the reason is worth recording, because it is a
   property of the data rather than of the model. Three front ends were
   tried on identical data: none (the original), a high-pass second
   difference, and direct LSB-plane extraction. All three sit at
   validation AUC ~0.50.

   The cause is measurable. The cover audio's LSB plane has lag-1
   autocorrelation +0.0007 and mean 0.5004 -- it is already
   indistinguishable from coin flips before anything is embedded, so LSB
   embedding has no structure to disturb. This is intrinsic to the signal:
   consecutive samples of a full-scale waveform at 16 kHz differ by
   thousands of quantization steps, which leaves the low bit
   equidistributed. Spatial image steganalysis works precisely because
   neighbouring pixels differ by only a few grey levels, making the low
   bit partly predictable. Real recordings recover some of this in quiet
   passages; these synthesized covers have none, as the amplitude envelope
   never drops below 0.15 of full scale.

   The high-pass front end is kept because it is the correct design and
   rules out "no residual extraction" as the explanation. The remaining
   failure is a dataset limitation and should be reported as one. It is
   corroborated independently by AudioSpectrogramCNN, which detects
   echo_hiding at 0.856 while scoring 0.339 on sample-domain LSB.
"""

import torch
import torch.nn as nn
import torchaudio


# ---------------------------------------------------------------------------
# Shared building blocks
# ---------------------------------------------------------------------------

class TLU(nn.Module):
    """Truncated linear unit: clamp(x, -T, +T).

    Bounds the residual so that a handful of strong edges cannot dominate
    the statistics the later layers see. Standard preprocessing in
    steganalysis CNNs.
    """

    def __init__(self, threshold=3.0):
        super().__init__()
        self.threshold = threshold

    def forward(self, x):
        return torch.clamp(x, -self.threshold, self.threshold)

    def extra_repr(self):
        return f"threshold={self.threshold}"


class AbsLayer(nn.Module):
    """|x|. Embedding noise is symmetric about zero, so its magnitude is
    the statistic that carries information; sign does not."""

    def forward(self, x):
        return torch.abs(x)


# ---------------------------------------------------------------------------
# Image model
# ---------------------------------------------------------------------------

# The classic KV (Ker-Vetterly) high-pass residual kernel used in SRM/XuNet-
# style steganalysis preprocessing. Highlights the kind of high-frequency
# noise pattern that spatial-domain embedding disturbs.
_KV_KERNEL = torch.tensor([
    [-1,  2, -2,  2, -1],
    [ 2, -6,  8, -6,  2],
    [-2,  8, -12, 8, -2],
    [ 2, -6,  8, -6,  2],
    [-1,  2, -2,  2, -1],
], dtype=torch.float32) / 12.0


class HighPassResidual(nn.Module):
    """Fixed (non-trainable) high-pass filter applied before the learned
    conv stack. This is what lets the network see embedding noise instead
    of image content.

    Input arrives as [0, 1] floats; we rescale to 0-255 so the residual is
    in grey-level units and the TLU threshold downstream means what it
    means in the literature.
    """

    def __init__(self, scale=255.0):
        super().__init__()
        self.register_buffer("kernel", _KV_KERNEL.view(1, 1, 5, 5))
        self.scale = scale

    def forward(self, x):
        return nn.functional.conv2d(x * self.scale, self.kernel, padding=2)


class ImageStegoCNN(nn.Module):
    def __init__(self, tlu_threshold=3.0):
        super().__init__()
        self.hpf = HighPassResidual()
        self.tlu = TLU(tlu_threshold)

        # Group 1 carries the ABS. See the module docstring: without it the
        # network throws away every negative residual.
        self.g1 = nn.Sequential(
            nn.Conv2d(1, 16, 5, padding=2),
            AbsLayer(),
            nn.BatchNorm2d(16),
            nn.Tanh(),
            nn.AvgPool2d(2),            # 256 -> 128
        )
        self.g2 = nn.Sequential(
            nn.Conv2d(16, 32, 5, padding=2),
            nn.BatchNorm2d(32), nn.Tanh(), nn.AvgPool2d(2),      # 128 -> 64
        )
        self.g3 = nn.Sequential(
            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True), nn.AvgPool2d(2),   # 64 -> 32
        )
        self.g4 = nn.Sequential(
            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(inplace=True), nn.AvgPool2d(2),  # 32 -> 16
        )
        self.g5 = nn.Sequential(
            nn.Conv2d(128, 128, 3, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(inplace=True), nn.AvgPool2d(2),  # 16 -> 8
        )

        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(64, 2),
        )

    def forward(self, x):
        x = self.tlu(self.hpf(x))
        x = self.g5(self.g4(self.g3(self.g2(self.g1(x)))))
        x = self.pool(x).flatten(1)
        return self.classifier(x)


# ---------------------------------------------------------------------------
# Audio models
# ---------------------------------------------------------------------------

class AudioSpectrogramCNN(nn.Module):
    """Log-mel-spectrogram CNN, the audio analogue of the image model above:
    a fixed, well-understood transform (mel spectrogram) first, then
    learned conv layers on top.

    Deliberately unchanged from the first full run: this is the model that
    worked, and keeping it fixed keeps that result comparable.
    """

    def __init__(self, sr=16000, n_mels=64):
        super().__init__()
        self.melspec = torchaudio.transforms.MelSpectrogram(
            sample_rate=sr, n_fft=512, hop_length=160, n_mels=n_mels
        )
        self.to_db = torchaudio.transforms.AmplitudeToDB()

        def block(c_in, c_out, pool=True):
            layers = [
                nn.Conv2d(c_in, c_out, 3, padding=1),
                nn.BatchNorm2d(c_out),
                nn.ReLU(inplace=True),
            ]
            if pool:
                layers.append(nn.AvgPool2d(2))
            return nn.Sequential(*layers)

        self.features = nn.Sequential(
            block(1, 16),
            block(16, 32),
            block(32, 64),
            block(64, 128),
        )
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(64, 2),
        )

    def forward(self, x):
        # x: [B, 1, T] raw waveform in, compute spectrogram here so the
        # dataset loader can stay simple and shared across model types
        spec = self.melspec(x)       # [B, 1, n_mels, frames]
        spec = self.to_db(spec)
        spec = (spec - spec.mean()) / (spec.std() + 1e-6)
        x = self.features(spec)
        x = self.pool(x).flatten(1)
        return self.classifier(x)


# Second difference. The audio counterpart of the KV kernel: it annihilates
# locally-linear signal and leaves sample-scale perturbation behind.
_AUDIO_HP_KERNEL = torch.tensor([1.0, -2.0, 1.0], dtype=torch.float32)


class AudioHighPassResidual(nn.Module):
    """Fixed high-pass front end for raw audio.

    Samples arrive as floats in [-1, 1], where a one-LSB change to 16-bit
    PCM is 1/32768, about 3e-5 relative -- utterly swamped by the carrier.
    We rescale to int16 units so a single-LSB perturbation is O(1), then
    take a second difference to suppress the smooth underlying signal and
    leave the perturbation standing.
    """

    def __init__(self, scale=32768.0):
        super().__init__()
        self.register_buffer("kernel", _AUDIO_HP_KERNEL.view(1, 1, 3))
        self.scale = scale

    def forward(self, x):
        return nn.functional.conv1d(x * self.scale, self.kernel, padding=1)


class AudioWaveformCNN(nn.Module):
    """1D CNN on the raw waveform, as an architectural comparison against
    the spectrogram model. Sample-domain embedding methods (LSB) should be
    more visible here, since computing a mel spectrogram smooths over
    single-sample perturbations.

    Now preceded by a fixed high-pass residual and TLU, mirroring the image
    model. Pooling is average rather than max: max pooling over a residual
    keeps only the largest deviation in each window and throws away the
    rest of the distribution, which is where the statistic lives.
    """

    def __init__(self, tlu_threshold=3.0):
        super().__init__()
        self.hpf = AudioHighPassResidual()
        self.tlu = TLU(tlu_threshold)

        def block(c_in, c_out, k=9, pool=4, act="relu", use_abs=False):
            layers = [nn.Conv1d(c_in, c_out, k, padding=k // 2)]
            if use_abs:
                layers.append(AbsLayer())
            layers.append(nn.BatchNorm1d(c_out))
            layers.append(nn.Tanh() if act == "tanh" else nn.ReLU(inplace=True))
            layers.append(nn.AvgPool1d(pool))
            return nn.Sequential(*layers)

        self.features = nn.Sequential(
            block(1, 16, act="tanh", use_abs=True),
            block(16, 32, act="tanh"),
            block(32, 64),
            block(64, 128),
            block(128, 128),
        )
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.classifier = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(64, 2),
        )

    def forward(self, x):
        x = self.tlu(self.hpf(x))
        x = self.features(x)
        x = self.pool(x).flatten(1)
        return self.classifier(x)


if __name__ == "__main__":
    img_model = ImageStegoCNN()
    dummy_img = torch.randn(4, 1, 256, 256).clamp(0, 1)
    out = img_model(dummy_img)
    print("ImageStegoCNN output shape:", out.shape)
    print("  params:", sum(p.numel() for p in img_model.parameters() if p.requires_grad))

    spec_model = AudioSpectrogramCNN()
    dummy_aud = torch.randn(4, 1, 32000).clamp(-1, 1)
    out = spec_model(dummy_aud)
    print("AudioSpectrogramCNN output shape:", out.shape)
    print("  params:", sum(p.numel() for p in spec_model.parameters() if p.requires_grad))

    wave_model = AudioWaveformCNN()
    out = wave_model(dummy_aud)
    print("AudioWaveformCNN output shape:", out.shape)
    print("  params:", sum(p.numel() for p in wave_model.parameters() if p.requires_grad))


In [ ]:
%%writefile src/classical_baseline.py
"""
Phase 3 - Classical baseline.

Deep CNNs need a benchmark to be compared against, otherwise there's no
evidence they're actually learning something a simpler method couldn't.
This implements a classical feature-based detector in the spirit of rich
models (SPAM/SRM-style): handcrafted statistical features computed from
the noise residual, fed into a shallow classifier (Random Forest / SVM
via scikit-learn).

Image features: co-occurrence-style statistics of the pixel difference
array (a lightweight stand-in for full SPAM features, which are more
involved than fits a course-scope classical baseline).

Audio features: statistics of the sample-difference signal plus basic
spectral flatness/centroid features, the audio analogue of the same idea.
"""

import os
import numpy as np
import pandas as pd
from PIL import Image
import soundfile as sf
from scipy.fft import rfft
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report

_PROJECT_ROOT = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))


# ---------------------------------------------------------------------------
# Feature extraction
# ---------------------------------------------------------------------------

def extract_image_features(img):
    """img: uint8 2D array. Returns a fixed-length feature vector built
    from the horizontal/vertical pixel-difference residual (where spatial-
    domain embedding noise shows up) plus a chi-square Pairs-of-Values
    statistic, the classical strong detector for LSB-style embedding:
    LSB embedding pulls adjacent histogram bin pairs (2k, 2k+1) toward
    equal frequency, which *lowers* this statistic relative to a natural
    image."""
    img = img.astype(np.float64)

    diff_h = np.diff(img, axis=1)
    diff_v = np.diff(img, axis=0)

    feats = []
    for d in (diff_h, diff_v):
        feats += [d.mean(), d.std(), d.var(),
                   np.mean(np.abs(d)), np.percentile(np.abs(d), 90)]
        sign_match = np.mean(np.sign(d[:, :-1]) == np.sign(d[:, 1:])) if d.shape[1] > 1 else 0.0
        feats.append(sign_match)

    lsb_plane = (img.astype(np.uint8) & 1)
    feats.append(lsb_plane.mean())

    feats += [img.mean(), img.std()]

    f = np.abs(rfft(img.flatten()))
    n = len(f)
    low_energy = f[: n // 8].sum()
    high_energy = f[n // 8:].sum()
    feats.append(high_energy / (low_energy + 1e-9))

    # chi-square pairs-of-values (log-transformed: raw values span several
    # orders of magnitude depending on image complexity)
    chi2 = _chi_square_pov(img.astype(np.uint8).flatten(), n_bins=256, vmin=0, vmax=256)
    feats.append(np.log1p(chi2))

    return np.array(feats, dtype=np.float64)


def _chi_square_pov(values_int, n_bins, vmin, vmax):
    hist, _ = np.histogram(values_int, bins=n_bins, range=(vmin, vmax))
    chi2 = 0.0
    for k in range(n_bins // 2):
        h0, h1 = hist[2 * k], hist[2 * k + 1]
        expected = (h0 + h1) / 2.0
        if expected > 0:
            chi2 += ((h0 - expected) ** 2) / expected + ((h1 - expected) ** 2) / expected
    return chi2


def extract_audio_features(sig):
    """sig: float32 1D array. Returns a fixed-length feature vector from
    the sample-difference residual, basic spectral stats, and a chi-square
    Pairs-of-Values statistic on the low byte of the 16-bit-quantized
    signal (the audio analogue of the image PoV feature above). Note:
    this feature is empirically much weaker for audio than for images in
    this dataset (see README / report), most likely because our synthetic
    covers (tones, chirps, FM synthesis) are lower-entropy than real
    speech/music recordings, this is an honest limitation worth reporting
    rather than a bug we should hide."""
    sig = sig.astype(np.float64)
    diff = np.diff(sig)

    feats = [diff.mean(), diff.std(), diff.var(),
              np.mean(np.abs(diff)), np.percentile(np.abs(diff), 90)]

    pcm = np.clip(sig * 32767, -32768, 32767).astype(np.int16)
    lsb_plane = pcm & 1
    feats.append(lsb_plane.mean())

    feats += [sig.mean(), sig.std()]

    spec = np.abs(rfft(sig))
    n = len(spec)
    low_energy = spec[: n // 8].sum()
    high_energy = spec[n // 8:].sum()
    feats.append(high_energy / (low_energy + 1e-9))

    spec_safe = spec + 1e-12
    geo_mean = np.exp(np.mean(np.log(spec_safe)))
    arith_mean = np.mean(spec_safe)
    feats.append(geo_mean / arith_mean)

    low_byte = (pcm.astype(np.int32) & 0xFF)
    chi2 = _chi_square_pov(low_byte, n_bins=256, vmin=0, vmax=256)
    feats.append(np.log1p(chi2))

    return np.array(feats, dtype=np.float64)


# ---------------------------------------------------------------------------
# Dataset assembly for classical models
# ---------------------------------------------------------------------------

def build_feature_matrix(manifest_df, modality):
    X, y, methods = [], [], []
    for _, row in manifest_df.iterrows():
        if modality == "image":
            data = np.array(Image.open(row.path))
            feat = extract_image_features(data)
        else:
            sig, _ = sf.read(row.path, dtype="float32")
            feat = extract_audio_features(sig)
        X.append(feat)
        y.append(0 if row.label == "cover" else 1)
        methods.append(row.method)
    return np.array(X), np.array(y), np.array(methods)


def run_classical_baseline(modality, exclude_methods=None):
    from dataset_loader import load_manifest, filter_manifest

    df = load_manifest()
    train_df = filter_manifest(df, modality, "train", exclude_methods=exclude_methods)
    test_df = filter_manifest(df, modality, "test")  # test on everything, incl. held-out method

    print(f"\n=== Classical baseline: {modality} "
          f"(excluded from training: {exclude_methods}) ===")

    # balance classes for training: with 9 stego variants per cover, a raw
    # 9:1 stego:cover ratio drowns out class_weight="balanced" in practice
    # and the model just learns to always predict "stego". Undersample
    # stego down to match the cover count instead.
    rng = np.random.default_rng(0)
    cover_rows = train_df[train_df.label == "cover"]
    stego_rows = train_df[train_df.label == "stego"]
    if len(stego_rows) > len(cover_rows):
        keep_idx = rng.choice(len(stego_rows), size=len(cover_rows), replace=False)
        stego_rows = stego_rows.iloc[keep_idx]
    train_df = pd.concat([cover_rows, stego_rows], ignore_index=True)

    print(f"Train (balanced): {len(train_df)} samples "
          f"({(train_df.label=='cover').sum()} cover / {(train_df.label=='stego').sum()} stego) "
          f"| Test: {len(test_df)} samples")

    X_train, y_train, _ = build_feature_matrix(train_df, modality)
    X_test, y_test, methods_test = build_feature_matrix(test_df, modality)

    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s = scaler.transform(X_test)

    clf = RandomForestClassifier(n_estimators=200, max_depth=12,
                                   class_weight="balanced", random_state=0, n_jobs=-1)
    clf.fit(X_train_s, y_train)

    y_pred = clf.predict(X_test_s)
    y_prob = clf.predict_proba(X_test_s)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)
    print(f"Overall test accuracy: {acc:.4f} | AUC: {auc:.4f}")

    # break down accuracy per method, this is the important number: if
    # accuracy on the held-out method is much lower than on seen methods,
    # that's expected and worth reporting; if it's close, that's evidence
    # of genuine blind generalization
    results_rows = []
    for m in sorted(set(methods_test)):
        mask = methods_test == m
        m_acc = accuracy_score(y_test[mask], y_pred[mask])
        results_rows.append({"modality": modality, "method": m,
                               "n": mask.sum(), "accuracy": m_acc})
        print(f"  {m:15s} n={mask.sum():4d}  accuracy={m_acc:.4f}")

    return pd.DataFrame(results_rows), clf, scaler


if __name__ == "__main__":
    results_dir = os.path.join(_PROJECT_ROOT, "results")
    os.makedirs(results_dir, exist_ok=True)

    all_results = []

    # baseline 1: train on all methods (standard blind setting)
    r_img, _, _ = run_classical_baseline("image", exclude_methods=None)
    r_img["setting"] = "all_methods_train"
    all_results.append(r_img)

    r_aud, _, _ = run_classical_baseline("audio", exclude_methods=None)
    r_aud["setting"] = "all_methods_train"
    all_results.append(r_aud)

    # baseline 2: held-out method test, train excluding dct_qim (image) /
    # echo_hiding (audio), the strongest / most different method in each
    # modality, to genuinely stress-test blindness
    r_img_ho, _, _ = run_classical_baseline("image", exclude_methods=["dct_qim"])
    r_img_ho["setting"] = "held_out_dct_qim"
    all_results.append(r_img_ho)

    r_aud_ho, _, _ = run_classical_baseline("audio", exclude_methods=["echo_hiding"])
    r_aud_ho["setting"] = "held_out_echo_hiding"
    all_results.append(r_aud_ho)

    final = pd.concat(all_results, ignore_index=True)
    out_path = os.path.join(results_dir, "classical_baseline_results.csv")
    final.to_csv(out_path, index=False)
    print(f"\nSaved classical baseline results to {out_path}")

In [ ]:
%%writefile src/train.py
"""
Phase 3 - CNN training script.

Trains ImageStegoCNN / AudioSpectrogramCNN / AudioWaveformCNN and reports
accuracy broken down by embedding method on the test set, which is what
lets us check the "blind" claim: if `--exclude-method` is set, that method
never appears during training, so its test accuracy tells us how well the
model generalizes to a completely unseen embedding algorithm.

Stability notes (why the defaults are what they are)
----------------------------------------------------
An earlier version of this script produced a model that scored 83.5%
overall test accuracy while getting only 8.3% of covers right, i.e. it had
collapsed to answering "stego" for almost everything and was being
rewarded for it by the ~90/10 stego/cover ratio in the test set. Its
validation accuracy also swung wildly between epochs (0.79 -> 0.16 -> 0.39
-> 0.19). Four things address that:

  1. Checkpoint selection uses BALANCED accuracy (the mean of cover recall
     and stego recall), so an always-stego model scores 0.5 and can never
     win the checkpoint.
  2. Default learning rate is 1e-4, not 1e-3. The steganalysis signal is a
     very low-amplitude residual; 1e-3 with Adam overshoots it and is the
     main source of the epoch-to-epoch thrashing.
  3. Gradient-norm clipping, for the same reason.
  4. BatchNorm momentum is lowered (--bn-momentum, default 0.01) so the
     running statistics used at eval time average over many more batches.
     At batch_size 16-32 with a signal this subtle, the PyTorch default of
     0.1 makes eval-time BN statistics noisy enough on its own to move
     validation accuracy by tens of points between epochs.

We also report AUC, which is threshold-independent and therefore the
honest headline number under this much class imbalance, and seed every
source of randomness (--seed) so runs reproduce exactly, matching the
guarantee Phase 2 data generation already makes.

Usage examples:
    python3 train.py --modality image --model image_cnn --epochs 20
    python3 train.py --modality image --model image_cnn --epochs 20 --exclude-method dct_qim
    python3 train.py --modality audio --model audio_spectrogram --epochs 20
    python3 train.py --modality audio --model audio_waveform --epochs 20 --exclude-method echo_hiding
"""

import os
import random
import argparse
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, WeightedRandomSampler
from sklearn.metrics import roc_auc_score

from dataset_loader import load_manifest, filter_manifest, ImageStegoDataset, AudioStegoDataset
from models import ImageStegoCNN, AudioSpectrogramCNN, AudioWaveformCNN

_PROJECT_ROOT = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
MODELS_DIR = os.path.join(_PROJECT_ROOT, "models")
RESULTS_DIR = os.path.join(_PROJECT_ROOT, "results")


def set_seed(seed):
    """Seed every source of randomness the training loop touches. Phase 2
    already guarantees the dataset itself is reproducible; this extends the
    same guarantee to training, which also makes the instability we were
    chasing reproducible rather than a moving target."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def make_weighted_sampler(df, generator=None):
    """Since covers are outnumbered ~9:1 by stego, use a weighted sampler
    so each training batch sees roughly balanced classes (undersampling
    the way classical_baseline.py does would throw away 90% of our
    generated stego data; a weighted sampler keeps all of it while still
    balancing what the model actually sees per batch)."""
    labels = (df.label != "cover").astype(int).values
    class_counts = np.bincount(labels, minlength=2).astype(np.float64)
    if (class_counts == 0).any():
        # only one class present -- nothing to balance, sample uniformly
        return None
    class_weights = 1.0 / class_counts
    sample_weights = class_weights[labels]
    return WeightedRandomSampler(sample_weights, num_samples=len(df),
                                 replacement=True, generator=generator)


def get_model(name, bn_momentum=None):
    if name == "image_cnn":
        model = ImageStegoCNN()
    elif name == "audio_spectrogram":
        model = AudioSpectrogramCNN()
    elif name == "audio_waveform":
        model = AudioWaveformCNN()
    else:
        raise ValueError(f"unknown model {name}")

    # Lower BatchNorm momentum => eval-time running statistics are averaged
    # over many more batches => far less epoch-to-epoch eval noise. See the
    # stability notes at the top of this file.
    if bn_momentum is not None:
        for m in model.modules():
            if isinstance(m, (nn.BatchNorm1d, nn.BatchNorm2d)):
                m.momentum = bn_momentum
    return model


def evaluate(model, loader, device):
    model.eval()
    all_probs, all_preds, all_labels, all_methods = [], [], [], []
    with torch.no_grad():
        for x, y, methods in loader:
            x = x.to(device)
            logits = model(x)
            probs = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
            preds = logits.argmax(dim=1).cpu().numpy()
            all_probs.extend(probs)
            all_preds.extend(preds)
            all_labels.extend(y.numpy())
            all_methods.extend(methods)

    all_probs = np.array(all_probs)
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    all_methods = np.array(all_methods)

    overall_acc = (all_preds == all_labels).mean()

    # cover/stego are heavily imbalanced in val/test (covers are ~10% of
    # samples), so raw accuracy can look good while the model just
    # defaults to predicting "stego" for almost everything. Balanced
    # accuracy (average of the two classes' recall) is the honest metric
    # and is what we use for checkpoint selection.
    cover_mask = all_labels == 0
    stego_mask = all_labels == 1
    cover_recall = (all_preds[cover_mask] == all_labels[cover_mask]).mean() if cover_mask.sum() else float("nan")
    stego_recall = (all_preds[stego_mask] == all_labels[stego_mask]).mean() if stego_mask.sum() else float("nan")
    balanced_acc = np.nanmean([cover_recall, stego_recall])

    # AUC is threshold-independent, so unlike accuracy it cannot be gamed
    # by a model that simply shifts its decision boundary toward the
    # majority class. Worth reporting alongside balanced accuracy.
    if cover_mask.sum() and stego_mask.sum():
        auc = roc_auc_score(all_labels, all_probs)
    else:
        auc = float("nan")

    per_method = {}
    for m in sorted(set(all_methods)):
        mask = all_methods == m
        per_method[m] = (all_preds[mask] == all_labels[mask]).mean()

    metrics = {
        "overall_acc": overall_acc,
        "balanced_acc": balanced_acc,
        "cover_recall": cover_recall,
        "stego_recall": stego_recall,
        "auc": auc,
    }
    return metrics, per_method


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--modality", choices=["image", "audio"], required=True)
    parser.add_argument("--model", choices=["image_cnn", "audio_spectrogram", "audio_waveform"], required=True)
    parser.add_argument("--epochs", type=int, default=20)
    parser.add_argument("--batch-size", type=int, default=32)
    parser.add_argument("--lr", type=float, default=1e-4)
    parser.add_argument("--seed", type=int, default=0)
    parser.add_argument("--patience", type=int, default=6,
                        help="stop early if val balanced accuracy hasn't improved for this many epochs")
    parser.add_argument("--min-epochs", type=int, default=8,
                        help="never early-stop before this epoch; a collapsed model sits at exactly "
                             "0.5 balanced accuracy for a while before it starts learning, and "
                             "stopping during that window kills the run before it begins")
    parser.add_argument("--bn-momentum", type=float, default=0.01,
                        help="BatchNorm momentum; lower = more stable eval-time statistics")
    parser.add_argument("--grad-clip", type=float, default=5.0,
                        help="max gradient norm (0 disables clipping)")
    parser.add_argument("--num-workers", type=int, default=-1,
                        help="DataLoader worker processes; -1 auto-selects (2 on GPU to keep it fed, "
                             "0 on CPU where decoding is not the bottleneck)")
    parser.add_argument("--exclude-method", type=str, default=None,
                        help="embedding method to hold out of training entirely, for the blindness test")
    parser.add_argument("--max-train-samples", type=int, default=None,
                        help="optional cap on training set size, for quick smoke tests")
    parser.add_argument("--tag", type=str, default=None,
                        help="run name for saved checkpoint/results, defaults to model+exclude info")
    args = parser.parse_args()

    set_seed(args.seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
    if device.type == "cuda":
        print(f"  GPU: {torch.cuda.get_device_name(0)}")

    df = load_manifest()
    exclude = [args.exclude_method] if args.exclude_method else None
    train_df = filter_manifest(df, args.modality, "train", exclude_methods=exclude)
    val_df = filter_manifest(df, args.modality, "val")   # validate on all methods
    test_df = filter_manifest(df, args.modality, "test")  # test on all methods (incl. held-out)

    if args.max_train_samples is not None and len(train_df) > args.max_train_samples:
        train_df = train_df.sample(args.max_train_samples, random_state=args.seed).reset_index(drop=True)

    print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")
    print(f"Excluded method (held out of training): {args.exclude_method}")
    print(f"Methods seen in training: {sorted(train_df.method.unique())}")

    DatasetCls = ImageStegoDataset if args.modality == "image" else AudioStegoDataset
    train_ds = DatasetCls(train_df)
    val_ds = DatasetCls(val_df)
    test_ds = DatasetCls(test_df)

    # On GPU the model step is fast enough that PNG/WAV decoding becomes the
    # bottleneck, so workers pay off. On CPU the convolutions dominate by an
    # order of magnitude and workers only add process overhead -- and on
    # Windows, without persistent_workers, they are respawned every epoch.
    num_workers = args.num_workers
    if num_workers < 0:
        num_workers = 2 if device.type == "cuda" else 0

    loader_kwargs = {"num_workers": num_workers}
    if num_workers > 0:
        loader_kwargs["persistent_workers"] = True
    if device.type == "cuda":
        loader_kwargs["pin_memory"] = True

    sampler_gen = torch.Generator()
    sampler_gen.manual_seed(args.seed)
    sampler = make_weighted_sampler(train_df, generator=sampler_gen)
    train_loader = DataLoader(train_ds, batch_size=args.batch_size, sampler=sampler,
                              shuffle=(sampler is None), **loader_kwargs)
    val_loader = DataLoader(val_ds, batch_size=args.batch_size, shuffle=False, **loader_kwargs)
    test_loader = DataLoader(test_ds, batch_size=args.batch_size, shuffle=False, **loader_kwargs)

    model = get_model(args.model, bn_momentum=args.bn_momentum).to(device)
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Model {args.model}: {n_params} trainable params "
          f"(lr={args.lr}, batch={args.batch_size}, bn_momentum={args.bn_momentum}, seed={args.seed})")

    optimizer = torch.optim.Adam(model.parameters(), lr=args.lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, args.epochs))
    criterion = nn.CrossEntropyLoss()

    history = []
    best_val_bal_acc = -1.0
    best_val_auc = -1.0
    best_state = None
    best_epoch = 0
    epochs_since_improvement = 0

    for epoch in range(1, args.epochs + 1):
        model.train()
        running_loss = 0.0
        n_batches = 0
        for x, y, _methods in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            if args.grad_clip and args.grad_clip > 0:
                nn.utils.clip_grad_norm_(model.parameters(), args.grad_clip)
            optimizer.step()
            running_loss += loss.item()
            n_batches += 1
        current_lr = optimizer.param_groups[0]["lr"]
        scheduler.step()

        train_loss = running_loss / max(1, n_batches)
        val_metrics, val_per_method = evaluate(model, val_loader, device)
        print(f"Epoch {epoch}/{args.epochs}  train_loss={train_loss:.4f}  "
              f"val_acc={val_metrics['overall_acc']:.4f}  "
              f"val_balanced_acc={val_metrics['balanced_acc']:.4f}  "
              f"val_auc={val_metrics['auc']:.4f}  "
              f"val_cover_recall={val_metrics['cover_recall']:.4f}  "
              f"val_stego_recall={val_metrics['stego_recall']:.4f}")
        history.append({"epoch": epoch, "train_loss": train_loss, "lr": current_lr, **val_metrics})

        # Checkpoint on BALANCED accuracy, not raw accuracy: with a ~90/10
        # stego/cover split, raw accuracy rewards a model that just defaults
        # to predicting "stego" for almost everything.
        #
        # AUC breaks ties, and that tiebreak matters more than it sounds. A
        # model that has collapsed to one constant prediction scores EXACTLY
        # 0.5000 balanced accuracy every epoch, so on a strict > comparison
        # nothing ever counts as an improvement, the patience counter runs
        # out, and the run is killed before it has learned anything. AUC
        # keeps moving while the model is still collapsed, so it tells us
        # whether the thing is actually making progress underneath.
        bal, auc = val_metrics["balanced_acc"], val_metrics["auc"]
        auc_cmp = -1.0 if auc is None or np.isnan(auc) else auc
        eps = 1e-9
        improved = (bal > best_val_bal_acc + eps) or \
                   (bal >= best_val_bal_acc - eps and auc_cmp > best_val_auc + eps)

        if improved:
            best_val_bal_acc = max(bal, best_val_bal_acc)
            best_val_auc = max(auc_cmp, best_val_auc)
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            best_epoch = epoch
            epochs_since_improvement = 0
            print(f"  -> new best (val_balanced_acc={bal:.4f}, val_auc={auc_cmp:.4f}), checkpointing")
        else:
            epochs_since_improvement += 1
            if args.patience and epoch >= args.min_epochs and epochs_since_improvement >= args.patience:
                print(f"  -> no improvement in {args.patience} epochs, stopping early at epoch {epoch}")
                break

    if best_state is not None:
        model.load_state_dict(best_state)
        print(f"\nRestored best checkpoint from epoch {best_epoch} "
              f"(val_balanced_acc={best_val_bal_acc:.4f}) for final evaluation")

    test_metrics, test_per_method = evaluate(model, test_loader, device)
    print(f"\nFinal test accuracy: {test_metrics['overall_acc']:.4f}  "
          f"(balanced: {test_metrics['balanced_acc']:.4f}, AUC: {test_metrics['auc']:.4f})")
    print(f"  cover recall: {test_metrics['cover_recall']:.4f}  "
          f"stego recall: {test_metrics['stego_recall']:.4f}")
    if test_metrics["cover_recall"] < 0.2:
        print("  WARNING: cover recall is very low -- this model is close to "
              "answering 'stego' for everything, so its per-method stego "
              "accuracies are NOT evidence of detection.")
    for m, acc in sorted(test_per_method.items()):
        flag = "  <-- HELD OUT (blind generalization test)" if m == args.exclude_method else ""
        print(f"  {m:15s} acc={acc:.4f}{flag}")

    os.makedirs(MODELS_DIR, exist_ok=True)
    os.makedirs(RESULTS_DIR, exist_ok=True)
    tag = args.tag or f"{args.model}_{'excl_' + args.exclude_method if args.exclude_method else 'allmethods'}"

    torch.save(model.state_dict(), os.path.join(MODELS_DIR, f"{tag}.pt"))

    pd.DataFrame(history).to_csv(os.path.join(RESULTS_DIR, f"train_history_{tag}.csv"), index=False)

    base = {"tag": tag, "modality": args.modality, "model": args.model,
            "excluded_method": args.exclude_method}
    test_rows = [dict(base, method=m, accuracy=acc) for m, acc in test_per_method.items()]
    test_rows.append(dict(base, method="OVERALL", accuracy=test_metrics["overall_acc"]))
    test_rows.append(dict(base, method="BALANCED", accuracy=test_metrics["balanced_acc"]))
    test_rows.append(dict(base, method="COVER_RECALL", accuracy=test_metrics["cover_recall"]))
    test_rows.append(dict(base, method="STEGO_RECALL", accuracy=test_metrics["stego_recall"]))
    test_rows.append(dict(base, method="AUC", accuracy=test_metrics["auc"]))
    results_path = os.path.join(RESULTS_DIR, f"test_results_{tag}.csv")
    pd.DataFrame(test_rows).to_csv(results_path, index=False)
    print(f"\nSaved model to models/{tag}.pt")
    print(f"Saved results to results/test_results_{tag}.csv")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/run_phase3.py
"""
Phase 3 - run every training configuration needed to close the phase out.

Six runs, which is what Phase 3 actually requires:

  image  / image_cnn          -- all methods          (standard blind setting)
  image  / image_cnn          -- held out dct_qim     (blindness stress test)
  audio  / audio_spectrogram  -- all methods
  audio  / audio_spectrogram  -- held out echo_hiding (blindness stress test)
  audio  / audio_waveform     -- all methods          (architecture comparison)
  audio  / audio_waveform     -- held out echo_hiding

Each run is a subprocess, so one crash doesn't take the whole batch down,
and a run whose results CSV already exists is skipped unless --force, so an
interrupted batch can simply be restarted. Everything is streamed to
results/phase3_runs.log as well as to the console, so progress can be
followed without watching the terminal.

    python3 run_phase3.py --smoke     # tiny capped runs, minutes, plumbing check
    python3 run_phase3.py             # the real thing
    python3 run_phase3.py --only image_cnn_allmethods
"""

import os
import sys
import time
import argparse
import datetime
import subprocess

_SRC_DIR = os.path.dirname(os.path.abspath(__file__))
_PROJECT_ROOT = os.path.dirname(_SRC_DIR)
RESULTS_DIR = os.path.join(_PROJECT_ROOT, "results")
LOG_PATH = os.path.join(RESULTS_DIR, "phase3_runs.log")

RUNS = [
    {"tag": "image_cnn_allmethods",             "modality": "image", "model": "image_cnn",          "exclude": None},
    {"tag": "image_cnn_excl_dct_qim",           "modality": "image", "model": "image_cnn",          "exclude": "dct_qim"},
    {"tag": "audio_spectrogram_allmethods",     "modality": "audio", "model": "audio_spectrogram",  "exclude": None},
    {"tag": "audio_spectrogram_excl_echo_hiding", "modality": "audio", "model": "audio_spectrogram", "exclude": "echo_hiding"},
    {"tag": "audio_waveform_allmethods",        "modality": "audio", "model": "audio_waveform",     "exclude": None},
    {"tag": "audio_waveform_excl_echo_hiding",  "modality": "audio", "model": "audio_waveform",     "exclude": "echo_hiding"},
]


def emit(line, log_f):
    sys.stdout.write(line)
    sys.stdout.flush()
    log_f.write(line)
    log_f.flush()


def run_one(cfg, args, log_f):
    tag = ("smoke_" + cfg["tag"]) if args.smoke else cfg["tag"]
    results_path = os.path.join(RESULTS_DIR, f"test_results_{tag}.csv")

    if os.path.exists(results_path) and not args.force:
        emit(f"\n[skip] {tag} -- results already exist (use --force to re-run)\n", log_f)
        return "skipped", 0.0

    cmd = [
        sys.executable, "-u", os.path.join(_SRC_DIR, "train.py"),
        "--modality", cfg["modality"],
        "--model", cfg["model"],
        "--epochs", str(args.epochs),
        "--batch-size", str(args.batch_size),
        "--lr", str(args.lr),
        "--seed", str(args.seed),
        "--patience", str(args.patience),
        "--min-epochs", str(args.min_epochs),
        "--num-workers", str(args.num_workers),
        "--tag", tag,
    ]
    if cfg["exclude"]:
        cmd += ["--exclude-method", cfg["exclude"]]
    if args.max_train_samples:
        cmd += ["--max-train-samples", str(args.max_train_samples)]

    emit(f"\n{'=' * 78}\n", log_f)
    emit(f"[run] {tag}  ({datetime.datetime.now():%Y-%m-%d %H:%M:%S})\n", log_f)
    emit(f"      {' '.join(cmd[1:])}\n", log_f)
    emit(f"{'=' * 78}\n", log_f)

    start = time.time()
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1, cwd=_SRC_DIR)
    for line in proc.stdout:
        emit(line, log_f)
    code = proc.wait()
    elapsed = time.time() - start

    status = "ok" if code == 0 else f"FAILED (exit {code})"
    emit(f"[done] {tag}: {status} in {elapsed / 60:.1f} min\n", log_f)
    return status, elapsed


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--smoke", action="store_true",
                        help="tiny capped runs written under smoke_* tags, to check the pipeline works")
    parser.add_argument("--epochs", type=int, default=None)
    parser.add_argument("--batch-size", type=int, default=32)
    parser.add_argument("--lr", type=float, default=1e-4)
    parser.add_argument("--seed", type=int, default=0)
    parser.add_argument("--patience", type=int, default=6)
    parser.add_argument("--min-epochs", type=int, default=8)
    parser.add_argument("--num-workers", type=int, default=-1,
                        help="-1 lets train.py auto-select by device")
    parser.add_argument("--max-train-samples", type=int, default=None)
    parser.add_argument("--only", type=str, default=None,
                        help="run just this tag (see RUNS at the top of this file)")
    parser.add_argument("--force", action="store_true",
                        help="re-run even if a results CSV already exists")
    args = parser.parse_args()

    # smoke defaults: small enough to finish in minutes, big enough to prove
    # the loop trains, evaluates, checkpoints and writes its outputs
    if args.smoke:
        if args.epochs is None:
            args.epochs = 2
        if args.max_train_samples is None:
            args.max_train_samples = 300
        args.patience = 0  # don't early-stop a 2-epoch run
    elif args.epochs is None:
        args.epochs = 20

    runs = RUNS if args.only is None else [r for r in RUNS if r["tag"] == args.only]
    if not runs:
        print(f"No run matches --only {args.only!r}. Valid tags: {[r['tag'] for r in RUNS]}")
        return 1

    os.makedirs(RESULTS_DIR, exist_ok=True)
    summary = []
    with open(LOG_PATH, "a", encoding="utf-8") as log_f:
        emit(f"\n\n########## phase 3 batch "
             f"({'SMOKE' if args.smoke else 'FULL'}, {datetime.datetime.now():%Y-%m-%d %H:%M:%S}) "
             f"##########\n", log_f)
        batch_start = time.time()
        for cfg in runs:
            status, elapsed = run_one(cfg, args, log_f)
            summary.append((cfg["tag"], status, elapsed))

        emit(f"\n{'=' * 78}\nBATCH SUMMARY "
             f"(total {(time.time() - batch_start) / 60:.1f} min)\n{'=' * 78}\n", log_f)
        for tag, status, elapsed in summary:
            emit(f"  {tag:40s} {status:20s} {elapsed / 60:6.1f} min\n", log_f)
        emit(f"\nLog written to {LOG_PATH}\n", log_f)

    failed = [t for t, s, _ in summary if s.startswith("FAILED")]
    return 1 if failed else 0


if __name__ == "__main__":
    sys.exit(main())


In [ ]:
%%writefile src/summarize_phase3.py
"""
Phase 3 - consolidate every result into one comparison table.

Each training run writes its own results/test_results_<tag>.csv, which makes
it impossible to see the thing Phase 3 is actually about: whether a detector
trained on some embedding methods still detects a method it has never seen.
This script joins the CNN runs with the classical baseline into
results/phase3_summary.csv and prints tables ready to paste into the README.

The subtlety it deliberately encodes: accuracy on a single stego method is
meaningless on its own, because a detector that answers "stego" to
everything scores 100% on every stego method and 0% on covers. So every
per-method number is reported next to that run's cover recall, and the
headline metrics are balanced accuracy and AUC rather than raw accuracy.

    python3 summarize_phase3.py
"""

import os
import glob
import numpy as np
import pandas as pd

_PROJECT_ROOT = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
RESULTS_DIR = os.path.join(_PROJECT_ROOT, "results")

SPECIAL_ROWS = {"OVERALL", "BALANCED", "COVER_RECALL", "STEGO_RECALL", "AUC"}


def _setting_name(excluded):
    return "all_methods_train" if not excluded else f"held_out_{excluded}"


def load_cnn_runs():
    """One record per trained model, from results/test_results_*.csv."""
    runs = []
    for path in sorted(glob.glob(os.path.join(RESULTS_DIR, "test_results_*.csv"))):
        tag = os.path.basename(path)[len("test_results_"):-len(".csv")]
        if tag.startswith("smoke_"):
            continue  # smoke runs are plumbing checks, not results
        df = pd.read_csv(path)
        if df.empty:
            continue

        first = df.iloc[0]
        excluded = first.get("excluded_method")
        excluded = None if pd.isna(excluded) else str(excluded)

        lookup = dict(zip(df.method, df.accuracy))
        per_method = {k: v for k, v in lookup.items() if k not in SPECIAL_ROWS and k != "none"}
        cover_recall = lookup.get("COVER_RECALL", lookup.get("none", np.nan))

        runs.append({
            "detector": f"cnn:{first.get('model', tag)}",
            "modality": first.get("modality", ""),
            "setting": _setting_name(excluded),
            "held_out_method": excluded,
            "tag": tag,
            "overall_acc": lookup.get("OVERALL", np.nan),
            "balanced_acc": lookup.get("BALANCED", np.nan),
            "cover_recall": cover_recall,
            "stego_recall": lookup.get("STEGO_RECALL", np.nan),
            "auc": lookup.get("AUC", np.nan),
            "per_method": per_method,
        })
    return runs


def load_classical_runs():
    """Rebuild the same record shape from the classical baseline CSV, which
    stores per-method accuracy and sample counts but no summary rows."""
    path = os.path.join(RESULTS_DIR, "classical_baseline_results.csv")
    if not os.path.exists(path):
        return []

    df = pd.read_csv(path)
    runs = []
    for (modality, setting), grp in df.groupby(["modality", "setting"]):
        cover_rows = grp[grp.method == "none"]
        stego_rows = grp[grp.method != "none"]

        cover_recall = cover_rows.accuracy.iloc[0] if len(cover_rows) else np.nan
        # weight each stego method by its sample count to get overall stego recall
        if len(stego_rows) and stego_rows.n.sum() > 0:
            stego_recall = float((stego_rows.accuracy * stego_rows.n).sum() / stego_rows.n.sum())
        else:
            stego_recall = np.nan
        overall_n = grp.n.sum()
        overall_acc = float((grp.accuracy * grp.n).sum() / overall_n) if overall_n else np.nan

        held_out = None
        if setting.startswith("held_out_"):
            held_out = setting[len("held_out_"):]

        runs.append({
            "detector": "classical:random_forest",
            "modality": modality,
            "setting": setting,
            "held_out_method": held_out,
            "tag": f"classical_{modality}_{setting}",
            "overall_acc": overall_acc,
            "balanced_acc": float(np.nanmean([cover_recall, stego_recall])),
            "cover_recall": cover_recall,
            "stego_recall": stego_recall,
            "auc": np.nan,  # not persisted per-method by classical_baseline.py
            "per_method": dict(zip(stego_rows.method, stego_rows.accuracy)),
        })
    return runs


def build_long_table(runs):
    """One row per (run, stego method), each carrying its run's cover recall
    so a per-method number is never read in isolation."""
    rows = []
    for r in runs:
        for method, acc in sorted(r["per_method"].items()):
            rows.append({
                "detector": r["detector"],
                "modality": r["modality"],
                "setting": r["setting"],
                "method": method,
                "is_held_out": method == r["held_out_method"],
                "method_recall": acc,
                "cover_recall": r["cover_recall"],
                "balanced_acc": r["balanced_acc"],
                "auc": r["auc"],
                "tag": r["tag"],
            })
    return pd.DataFrame(rows)


def _fmt(v, nd=3):
    return "n/a" if v is None or (isinstance(v, float) and np.isnan(v)) else f"{v:.{nd}f}"


def print_run_table(runs):
    print("\n### Detector performance (test set)\n")
    print("| Detector | Modality | Setting | Balanced acc | Cover recall | Stego recall | AUC |")
    print("|---|---|---|---|---|---|---|")
    for r in sorted(runs, key=lambda r: (r["modality"], r["detector"], r["setting"])):
        print(f"| {r['detector']} | {r['modality']} | {r['setting']} | "
              f"{_fmt(r['balanced_acc'])} | {_fmt(r['cover_recall'])} | "
              f"{_fmt(r['stego_recall'])} | {_fmt(r['auc'])} |")


def print_blindness_table(runs):
    """The Phase 3 headline: for each held-out run, how well the detector
    does on the method it never saw versus the methods it did see."""
    print("\n### Blind generalization (held-out method vs. seen methods)\n")
    print("| Detector | Modality | Held-out method | Acc on held-out | "
          "Mean acc on seen methods | Cover recall |")
    print("|---|---|---|---|---|---|")
    for r in sorted(runs, key=lambda r: (r["modality"], r["detector"])):
        if not r["held_out_method"]:
            continue
        held = r["per_method"].get(r["held_out_method"], np.nan)
        seen = [v for k, v in r["per_method"].items() if k != r["held_out_method"]]
        seen_mean = float(np.mean(seen)) if seen else np.nan
        print(f"| {r['detector']} | {r['modality']} | {r['held_out_method']} | "
              f"{_fmt(held)} | {_fmt(seen_mean)} | {_fmt(r['cover_recall'])} |")


def warn_degenerate(runs):
    bad = [r for r in runs
           if not np.isnan(r["cover_recall"]) and r["cover_recall"] < 0.2]
    if bad:
        print("\n!! Degenerate-detector warning\n")
        print("These runs have cover recall below 0.2, meaning they answer "
              "'stego' to almost everything.\nTheir per-method stego "
              "accuracies are NOT evidence of detection and should not be "
              "reported as such:\n")
        for r in bad:
            print(f"  - {r['tag']}: cover recall {r['cover_recall']:.3f}, "
                  f"balanced acc {r['balanced_acc']:.3f}")


def main():
    runs = load_cnn_runs() + load_classical_runs()
    if not runs:
        print("No results found in results/. Run run_phase3.py first.")
        return

    long_df = build_long_table(runs)
    out_path = os.path.join(RESULTS_DIR, "phase3_summary.csv")
    long_df.to_csv(out_path, index=False)

    run_df = pd.DataFrame([{k: v for k, v in r.items() if k != "per_method"} for r in runs])
    run_out = os.path.join(RESULTS_DIR, "phase3_summary_by_run.csv")
    run_df.to_csv(run_out, index=False)

    print_run_table(runs)
    print_blindness_table(runs)
    warn_degenerate(runs)

    print(f"\nSaved {out_path}")
    print(f"Saved {run_out}")


if __name__ == "__main__":
    main()


## 4 · Build the dataset

Regenerates 1500 cover images and 1500 cover audio clips, then every embedding
method at every payload rate — 30,000 samples — and writes the manifest with its
cover-level train/val/test split.

Up from 400 because the first full run was limited by cover diversity as much as
by architecture: only 280 covers reached the training split and all 2,520 stego
samples derived from those same 280. Crops are free. It is still capped by the
15 source photographs, which is the honest limitation for the write-up.

Takes a few minutes, almost all of it writing 30,000 files to disk. (The
LSB-matching and DCT-QIM embedders used to be Python loops and made this a
25-minute step; they're vectorised now, with byte-identical output.)

In [ ]:
%cd /content/chameleon/src
!python data_generation.py
!python build_dataset.py

## 5 · Integrity check

Confirms paths resolve, every file exists, and the held-out filter actually removes the method.

In [ ]:
!python dataset_loader.py

## 6 · Classical baseline

Handcrafted features plus a random forest, the benchmark the CNNs have to beat.
Runs on CPU and takes roughly 5–15 minutes. The dataset changed when the cover
generator was fixed, so the old numbers no longer apply and this has to be rerun.

In [ ]:
!python classical_baseline.py

## 7 · The six training runs

Image and audio, each with an all-methods detector and a held-out-method
detector, plus both audio architectures for the spectrogram-vs-waveform
comparison.

25 epochs rather than 40: each epoch now covers 10,500 training samples instead
of 2,800, so the number of gradient steps per epoch has roughly quadrupled.
Early stopping and best-epoch checkpointing make overshooting cheap anyway.

What to expect, from the controlled A/B that motivated the architecture change:

- **image_cnn should now work.** The ABS + TLU revision took validation AUC from
  0.586 to 0.722 on identical data and budget, against 0.504 balanced / 0.516
  AUC in the previous full run.
- **audio_spectrogram should stay roughly where it was** (0.653 balanced /
  0.707 AUC). It is deliberately unchanged, as the control.
- **audio_waveform is expected to stay at chance.** Three front ends were tested
  and all sit at 0.50. The cover audio's LSB plane has autocorrelation +0.0007
  before anything is embedded, so there is no structure for sample-domain LSB to
  disturb. That is a dataset property and belongs in the report as a finding,
  not a bug to chase further.

In [ ]:
!python run_phase3.py --epochs 25 --patience 8 --min-epochs 10

## 8 · Consolidated results

The tables below are formatted for pasting straight into the README.

In [ ]:
!python summarize_phase3.py

## 9 · Download

Results and trained checkpoints. The CSVs are what matter; the zip is small.

In [ ]:
%cd /content/chameleon
!zip -qr phase3_results.zip results models
print("size:", round(os.path.getsize('phase3_results.zip')/1e6, 1), "MB")
from google.colab import files
files.download('phase3_results.zip')